# Cincinatti car crashes

## Dataset definition

All reported Cincinnati, Ohio car crashes since 2010.

This dataset includes fatal, injury, and non-injury crashes. In compliance with privacy laws, all Public Safety datasets are anonymized

[Original dataset](https://data.cincinnati-oh.gov/safety/Traffic-Crash-Reports-CPD-/rvmt-pkmq/about_data)

**Data Dictionary:**
![image](Data-Dictionary.png)


### Inspiration
- Find which areas of town are the sources of most crashes?
- Figure out how much weather changes the frequency of crashes.
- Predecir accidentes datas en una comunidad dadas unas condiciones.

## 1. Contextualización del problema y descripción del dataset

## 2. Objetivos principales

In [ ]:
# # Instalar dependencias
# !pip install kagglehub
# !pip install rich
# !pip install pandas


: 

In [ ]:
import os
import kagglehub
from rich import print
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

SEED=1234


In [ ]:
# configuracion de API de Kaggle
os.environ["KAGGLE_USERNAME"] = "user"
os.environ["KAGGLE_KEY"] = "token"


In [ ]:
# Descarga del dataset
path = kagglehub.dataset_download("steverusso/cincinnati-car-crash-data")
print("Path to dataset files:", path)

print("Descargando el dataset")
dataset = pd.read_csv(path+"/cincinnati_traffic_crash_data__cpd.csv") # Carga datos desde un CSV y devuelve un DataFrame
# dataset.to_csv("archivo.csv", index=False)

dataset.head()

In [ ]:
# Exploracion de datos

print("Info")
print(dataset.info())
print("Valores nulos")
print(dataset.isnull().sum())

## 3. Analítica de datos

## 4. Limpieza inicial de variables

### 4.1  Variables de ID de registros
- La columna ```Unnamed: 0``` corresponde al número de fila, luego la eliminamos porque no aporta información.
- Las columnas ```INSTANCEID``` y ```LOCALREPORTNO``` se refieren al identificador de una incidencia de accidente y al identificador del informe del accidente, respectivamente. No queremos entrenar sobre valores que se refieran a registros.

Se eliminan estas columnas:

In [ ]:
print(f"Tamaño del dataset original: {dataset.shape}")
dataset = dataset.drop(columns=["Unnamed: 0","INSTANCEID", "LOCALREPORTNO"])
print(f"Tamaño del dataset tras eliminar columnas de ID de registros: {dataset.shape}")

### 4.2  Variables de ubicación redundantes

Las siguientes variables sirven para ubicar geográficamente el lugar del incidente. Todas ellas hacen referencia a la ubicación donde ha ocurrido el accidente, pero lo hacen con distintos niveles de precisión. Es conviene reducir redundancia para quedarse con las que aporten más información.

- ```ADDRESS_X```: dirección del accidente, anonimizada a nivel de bloque.
- ```LATITUDE_X```: coordenada geográfica de latitud.
- ```LONGITUDE_X```: coordenada geográfica de longitud.
- ```CPD_NEIGHBORHOOD```: barrio según la división de la policía (Cincinnati Police Department).
- ```SNA_NEIGHBORHOOD```: barrio estadístico de la ciudad de Cincinnati.
- ```ZIP```: código postal.

In [ ]:
geo_columns = ["ADDRESS_X", "CPD_NEIGHBORHOOD", "ZIP", "SNA_NEIGHBORHOOD"]

# Cantidad de valores únicos
for col in geo_columns:
    print(f"{col}: {dataset[col].nunique()} valores únicos")

# Figura con 2 filas y 2 columnas
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()  # para iterar fácilmente

for i, col in enumerate(geo_columns):
    # Top 20 valores más frecuentes
    top_values = dataset[col].value_counts().head(20)

    axes[i].bar(top_values.index.astype(str), top_values.values)
    axes[i].set_title(f"Top 20 valores de {col}", fontsize=14)
    axes[i].set_xlabel(col, fontsize=12)
    axes[i].set_ylabel("Frecuencia", fontsize=12)
    axes[i].tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.show()

En primer lugar, se debe descartar la columna ```ADDRESS_X``` ya que sus valores han sido modificados y toma demasiados valores categóricos únicos para que merezca la pena hacer un _encoding_ sobre ellos.

A continuación, parece razonable mantener las coordenadas geográficas ya que son valores de tipo _float_, que podrán ser usados con facilidad en el proceso de entrenamiento.

Finalmente, las variables ```CPD_NEIGHBORHOOD```, ```SNA_NEIGHBORHOOD```, ```COMMUNITY_COUNCIL_NEIGHBORHOOD``` y ```ZIP``` representan agrupaciones dadas ciertas coordenadas geográficas. Puesto que todas ellas representan un tipo de información similar, tomamos inicialmente ```SNA_NEIGHBORHOOD``` para los análisis, aunque podría ser interesante repetir los resultados obtenidos para las otras. Se ha escogido esta principalmente porque la ciudad de Cincinatti ya había hehco una agrupación estadística de los barrios, además de que tiene pocas categorías únicas.

In [ ]:
dataset = dataset.drop(columns=["ADDRESS_X","CPD_NEIGHBORHOOD", "COMMUNITY_COUNCIL_NEIGHBORHOOD", "ZIP"])
print(f"Tamaño del dataset tras eliminar columnas de variables de ubicación redundantes: {dataset.shape}")

### 4.3  Variables categóricas idénticas

Se ha observado que las variables ```CRASHSEVERITYID``` y ```CRASHSEVERITY``` representan la misma información, ya que existe una correspondencia uno a uno entre ambas. Se decide eliminar la de ```CRASHSEVERITYID``` porque se empleará la otra para el mapa de valores de severidad que se creará más adelante. Realmente es indiferente cuál de las dos eliminar, pero se conserva ```CRASHSEVERITY``` por ser más clara para esta priemra fase.

In [ ]:
# identificar relación uno a uno de los valores
print(dataset.groupby("CRASHSEVERITY")["CRASHSEVERITYID"].unique())
print(dataset.groupby("CRASHSEVERITY")["CRASHSEVERITYID"].nunique())

dataset = dataset.drop(columns=["CRASHSEVERITYID"])
print(f"Tamaño del dataset tras eliminar la columna CRASHSEVERITYID: {dataset.shape}")

Tras esta limpieza inicial hemos pasado de 27 variables (columnas) a 20, que, en prinicipio, cuentan con información útil para entrenar los modelos sobre sus instancias.

In [ ]:
dataset = dataset.drop_duplicates()
print(f"Tamaño del dataset tras eliminar filas duplicadas: {dataset.shape}")

print("Info actualizada:")
print(dataset.info())
print("Valores nulos actualizado:")
print(dataset.isnull().sum())

dataset.head()

## 5. Transformación de los datos

Una vez seleccionadas todas aquellas columnas que, a priori, serán útiles para el análisis, se procede a la trasnformación del tipo de dato para que puedan ser empleados por los diferentes modelos que se desarrollen.

Se observa que hay gran cantidad de columnas con valores nominales, por lo que será necesario valorar el tipo de transformación según el tipo de valores nominales que tomen esas variables, si admiten orden o no.

En primer lugar, las columnas `CRASHDATE` y `DATECRASHREPORTED` se refieren a fechas, por lo que se transforman al tipo de dato de `datetime` para que los modelos puedan aprovechar esta información temporal.

Luego, las siguientes variables son de tipo nominal y sin posibilidad de establecer un orden:
- `CRASHLOCATION`:
- `MANNEROFCRASH`:
- `UNITTYPE`:
- `GENDER`:
- `SNA_NEIGHBORHOOD`:
- `TYPEOFPERSON`:

Se empleará la transformación de _One-Hot Encoding_ para poder trabajar con los valores de estas variables.

Finalmente, las siguientes varibles tienen valores nominales pero admiten un orden.
- `AGE`:
- `CRASHSEVERITY`:
- `INJURIES`:
- `LIGHTCONDITIONSPRIMARY`:
- `ROADSURFACE`:
- `WEATHER`:
- `DAYOFWEEK`:
- `ROADCONDITIONSPRIMARY`:
- `ROADCONTOUR`:

Se procede con un _Ordinal Encoding_ para poder asignar un valor numérico que respete este orden. Para cada variable, se asignará manualmente un valor de manera ascendente a cada categoría.



In [ ]:
# transformar fechas a datetime
dataset["CRASHDATE"] = pd.to_datetime(dataset["CRASHDATE"], format="%m/%d/%Y %I:%M:%S %p")
dataset["DATECRASHREPORTED"] = pd.to_datetime(dataset["DATECRASHREPORTED"], format="%m/%d/%Y %I:%M:%S %p")

In [ ]:
numeric_cols = [
    "AGE",
    "CRASHSEVERITY",
    "INJURIES",
    "LIGHTCONDITIONSPRIMARY",
    "ROADSURFACE",
    "WEATHER",
    "DAYOFWEEK",
    "ROADCONDITIONSPRIMARY",
    "ROADCONTOUR"
]

OneHotEncoding_cols = [
    "CRASHLOCATION",
    "MANNEROFCRASH",
    "UNITTYPE",
    "GENDER",
    "SNA_NEIGHBORHOOD",
    "TYPEOFPERSON"
]


# fig, axes = plt.subplots(5, 2, figsize=(20, 15))
# axes = axes.flatten()

# for i, col in enumerate(numeric_cols):
#     dataset[col].value_counts().plot(
#         kind="pie",
#         ax=axes[i],
#         autopct="%1.1f%%"
#     )
#     axes[i].set_title(col)
#     axes[i].set_ylabel("")

# # eliminar subplots vacíos
# for j in range(i+1, len(axes)):
#     fig.delaxes(axes[j])

# plt.suptitle("Distribución CATEGÓRICA", fontsize=16)
# plt.tight_layout()
# plt.show()

# fig, axes = plt.subplots(2, 2, figsize=(20, 15))
# axes = axes.flatten()

# for i, col in enumerate(OneHotEncoding_cols):
#     dataset[col].value_counts().plot(
#         kind="pie",
#         ax=axes[i],
#         autopct="%1.1f%%"
#     )
#     axes[i].set_title(col)
#     axes[i].set_ylabel("")

# # eliminar subplots vacíos
# for j in range(i+1, len(axes)):
#     fig.delaxes(axes[j])

# plt.suptitle("Distribución CATEGÓRICA", fontsize=16)
# plt.tight_layout()
# plt.show()

In [ ]:
# variables nominales con orden
AGE_map = {
    'UNDER 18': 0,
    '18-25': 1,
    '26-30': 2,
    '31-40': 3,
    '41-50': 4,
    '51-60': 5,
    '61-70': 6,
    'OVER 70': 7,
    'UNKNOWN': np.nan
}

CRASHSEVERITY_map = {
    '5 - PROPERTY DAMAGE ONLY': 0,
    '3 - PROPERTY DAMAGE ONLY (PDO)': 0,
    '4 - INJURY POSSIBLE': 1,
    '3 - MINOR INJURY SUSPECTED': 2,
    '2 - INJURY': 3,
    '2 - SERIOUS INJURY SUSPECTED': 4,
    '1 - FATAL INJURY': 5,
    '1 - FATAL': 6
}

INJURIES_map = {
    '1 - NO INJURY / NONE REPORTED': 0,
    '5 - NO APPARENTY INJURY': 1,
    '4 - POSSIBLE INJURY': 2,
    '2 - POSSIBLE': 3,
    '3 - NON-INCAPACITATING': 4,
    '3 - SUSPECTED MINOR INJURY': 5,
    '4 - INCAPACITATING': 6,
    '2 - SUSPECTED SERIOUS INJURY': 7,
    '5 - FATAL': 8,
    '1 - FATAL': 8
}

LIGHTCONDITIONSPRIMARY_map = {
    '1 - DAYLIGHT': 0,
    '2 - DAWN': 1,
    '2 - DUSK': 2,
    '3 - DUSK': 2,
    '3 - DARK - LIGHTED ROADWAY': 3,
    '4 - DARK - LIGHTED ROADWAY': 3,
    '4 - DARK – ROADWAY NOT LIGHTED': 4,
    '5 - DARK – ROADWAY NOT LIGHTED': 4,
    '5 - DARK – UNKNOWN ROADWAY LIGHTING': 5,
    '6 - DARK – UNKNOWN ROADWAY LIGHTING': 5,
    '8 - OTHER': np.nan,
    '9 - OTHER': np.nan,
    '9 - UNKNOWN': np.nan
}

ROADSURFACE_map = {
    '1 - CONCRETE': 0,
    '2 - BLACKTOP, BITUMINOUS, ASPHALT': 1,
    '3 - BRICK/BLOCK': 2,
    '4 - SLAG, GRAVEL, STONE': 3,
    '5 - DIRT': 4,
    '6 - OTHER': np.nan,
    '9 - OTHER': np.nan,
    '9 - UNKNOWN': np.nan
}

WEATHER_map = {
    '1 - CLEAR': 0,
    '2 - CLOUDY': 1,
    '3 - FOG, SMOG, SMOKE': 2,
    '4 - RAIN': 3,
    '5 - SLEET, HAIL': 4,
    '5 - SLEET,HAIL': 4,
    '6 - SNOW': 5,
    '7 - SEVERE CROSSWINDS': 6,
    '8 - BLOWING SAND, SOIL, DIRT, SNOW': 7,
    '9 - FREEZING RAIN OR FREEZING DRIZZLE': 8,
    '9 - OTHER/UNKNOWN': np.nan,
    '99 - OTHER/UNKNOWN': np.nan
}

DAYOFWEEK_map = {
    'MON': 0,
    'TUE': 1,
    'WED': 2,
    'THU': 3,
    'FRI': 4,
    'SAT': 5,
    'SUN': 6
}

ROADCONDITIONPRIMARY_map = {
    '01 - DRY': 0,
    '02 - WET': 1,
    '07 - SLUSH': 2,
    '03 - SNOW': 3,
    '04 - ICE': 4,
    '06 - WATER (STANDING, MOVING)': 5,
    '05 - SAND, MUD, DIRT, OIL, GRAVEL': 6,
    '10 - OTHER': np.nan,
    '09 - OTHER': np.nan,
    '09 - UNKNOWN': np.nan,
    '99 - UNKNOWN': np.nan
}

ROADCONTOUR_map = {
    '1 - STRAIGHT LEVEL': 0,
    '2 - STRAIGHT GRADE': 1,
    '3 - CURVE LEVEL': 2,
    '4 - CURVE GRADE': 3,
    '9 - UNKNOWN': np.nan
}

dataset["AGE"] = dataset["AGE"].map(AGE_map)
dataset["CRASHSEVERITY"] = dataset["CRASHSEVERITY"].map(CRASHSEVERITY_map)
dataset["INJURIES"] = dataset["INJURIES"].map(INJURIES_map)
dataset["LIGHTCONDITIONSPRIMARY"] = dataset["LIGHTCONDITIONSPRIMARY"].map(LIGHTCONDITIONSPRIMARY_map)
dataset["ROADSURFACE"] = dataset["ROADSURFACE"].map(ROADSURFACE_map)
dataset["WEATHER"] = dataset["WEATHER"].map(WEATHER_map)
dataset["DAYOFWEEK"] = dataset["DAYOFWEEK"].map(DAYOFWEEK_map)
dataset["ROADCONDITIONSPRIMARY"] = dataset["ROADCONDITIONSPRIMARY"].map(ROADCONDITIONPRIMARY_map)
dataset["ROADCONTOUR"] = dataset["ROADCONTOUR"].map(ROADCONTOUR_map)

In [ ]:
# unificar valores de la columna GENDER
dataset["GENDER"] = dataset["GENDER"].replace({
    "MALE": "M - MALE",
    "FEMALE": "F - FEMALE"
})

# sustituir todos los valores de UNKNOWN por NaN
for col in OneHotEncoding_cols:
    dataset[col] = dataset[col].astype(str).str.strip()
    mask = dataset[col].str.contains("unknown", case=False, na=False)
    dataset.loc[mask, col] = np.nan

# aplicar One Hot Encoding a las columnas categóricas que no admiten orden
dataset = pd.get_dummies(dataset, columns=OneHotEncoding_cols, dummy_na=True)

In [ ]:
# Info dataset tras el procesado de variables categóricas
print("\nValores nulos (NaN) en columnas categoricas:")
print(dataset[numeric_cols].isna().sum())

print("Info")
print(dataset.info())

print("Columnas del dataset:")
print(list(dataset.columns))

### 5.1 Data imputation

In [ ]:
# eliminar nan (naive)
previous_shape = dataset.shape[0]
print("Tamaño antes de eliminar NaN:", previous_shape)
dataset = dataset.dropna()
new_shape = dataset.shape[0]
print("Tamaño después de eliminar NaN:", new_shape)
print("Filas eliminadas:", previous_shape - new_shape)

### 5.2 División de los datos

In [ ]:
# 70% trainset, 20% testset, 10% validationset
trainset, testset = train_test_split(dataset, test_size=0.3, train_size=0.7, random_state=SEED, shuffle=True)
testset, validationset = train_test_split(testset, test_size=2/3, train_size=1/3, random_state=SEED, shuffle=True)

## 6. Modelo 1: Random Forest

En este bloque se importan las librerías necesarias para el modelado:

- **`RandomForestClassifier`**: el clasificador de Random Forest de scikit-learn.
- **`GridSearchCV`**, **`cross_val_score`** y **`KFold`**: para la búsqueda de hiperparámetros y la validación cruzada.
- **Métricas**: `accuracy_score`, `precision_score`, `recall_score`, `f1_score`, `confusion_matrix` y `classification_report` para evaluar los resultados.
- **`seaborn`**: para visualizar la matriz de confusión con un mapa de calor.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score, KFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


### 6.1. Preparación de los conjuntos de datos para el modelo

Antes de entrenar, hay que separar las variables de entrada (X) de la variable a predecir (y) en cada uno de los tres conjuntos.

La variable objetivo es `CRASHSEVERITY`. Las columnas `CRASHDATE` y `DATECRASHREPORTED` no se pueden usar directamente en formato datetime, pero antes de descartarlas se extraen dos características temporales: la **hora del accidente** (`CRASH_HOUR`) y el **mes** (`CRASH_MONTH`). La hora puede capturar diferencias entre hora punta y madrugada, y el mes recoge variaciones estacionales en las condiciones de conducción (hielo en invierno, más tráfico en verano). `DATECRASHREPORTED` se descarta porque la única información útil sería la diferencia respecto a `CRASHDATE`, lo que se deja como posible mejora futura.

También hay que asegurarse de que el conjunto de validación no se usa en ningún momento durante el entrenamiento ni la optimización, para que sus métricas reflejen el rendimiento real sobre datos completamente nuevos.

In [ ]:
# Ingeniería de variables: extraer hora y mes de CRASHDATE antes de descartarla
def add_time_features(df):
    d = df.copy()
    d['CRASH_HOUR'] = d['CRASHDATE'].dt.hour
    d['CRASH_MONTH'] = d['CRASHDATE'].dt.month
    return d

trainset = add_time_features(trainset)
testset = add_time_features(testset)
validationset = add_time_features(validationset)

# Definir variable objetivo
target_column = 'CRASHSEVERITY'

if target_column not in trainset.columns:
    print(f"Columna '{target_column}' no encontrada")
    print(f"Columnas disponibles: {trainset.columns.tolist()}")
else:
    # Separar features y target en cada set
    y_train = trainset[target_column]
    X_train = trainset.drop(columns=[target_column, "CRASHDATE", "DATECRASHREPORTED"])

    y_test = testset[target_column]
    X_test = testset.drop(columns=[target_column, "CRASHDATE", "DATECRASHREPORTED"])

    y_val = validationset[target_column]
    X_val = validationset.drop(columns=[target_column, "CRASHDATE", "DATECRASHREPORTED"])

    print(f"Conjunto de Entrenamiento (Train): {X_train.shape[0]} muestras, {X_train.shape[1]} features")
    print(f"Conjunto de Test: {X_test.shape[0]} muestras, {X_test.shape[1]} features")
    print(f"Conjunto de Validación: {X_val.shape[0]} muestras, {X_val.shape[1]} features")

    print("--- Distribución de clases en Train ---")
    print(y_train.value_counts().sort_index())
    print("Proporción:")
    print((y_train.value_counts(normalize=True).sort_index() * 100).round(2))


El primer modelo que se va a entrenar es un **Random Forest**. Se ha elegido este algoritmo por varias razones prácticas para este problema:

- No requiere normalizar los datos. Los modelos basados en árboles toman decisiones por umbrales de valor, por lo que la escala de las variables no afecta al resultado. Esto simplifica el pipeline dado que el dataset contiene variables de tipos muy distintos (ordinales, binarias del one-hot encoding, coordenadas...).
- Al combinar múltiples árboles entrenados sobre subconjuntos aleatorios, es resistente al sobreajuste.
- Genera directamente la importancia de cada variable, lo que ayuda a entender qué factores influyen más en la severidad del accidente.

El proceso se divide en tres pasos: entrenar un modelo inicial con parámetros por defecto (*baseline*), evaluar su estabilidad con validación cruzada y, por último, buscar una mejor configuración con GridSearchCV.

### 6.2. Modelo Baseline (sin optimización de hiperparámetros)

Lo primero es entrenar el modelo con los parámetros por defecto, sin ningún ajuste. Esto sirve como referencia para saber si el algoritmo ya funciona razonablemente bien con este dataset y para cuantificar después cuánto mejora la optimización.

In [ ]:
import time

# Crear modelo baseline
model_forest_baseline = RandomForestClassifier(
    random_state=SEED,
    n_jobs=-1
)

# Entrenar
print("Entrenando modelo baseline...")
start_time = time.time()
model_forest_baseline.fit(X_train, y_train)
elapsed = time.time() - start_time
print(f"Entrenamiento completado en {elapsed:.2f} segundos")

# Predicciones
y_pred_baseline_train = model_forest_baseline.predict(X_train)
y_pred_baseline_test = model_forest_baseline.predict(X_test)
y_pred_baseline_val = model_forest_baseline.predict(X_val)

# Evaluación
print("--- Resultados BASELINE ---")
acc_baseline_train = accuracy_score(y_train, y_pred_baseline_train)
acc_baseline_test = accuracy_score(y_test, y_pred_baseline_test)
acc_baseline_val = accuracy_score(y_val, y_pred_baseline_val)

print(f"Train Accuracy: {acc_baseline_train:.4f}")
print(f"Test Accuracy:  {acc_baseline_test:.4f}")
print(f"Val Accuracy:   {acc_baseline_val:.4f}")

# Visualizar gap overfitting
gap = acc_baseline_train - acc_baseline_test
print(f"Gap Train-Test: {gap:.4f}", end="")
if gap > 0.15:
    print("(Overfitting moderado)")
elif gap > 0.05:
    print(" ✓ (Normal para Random Forest)")
else:
    print(" ✓ (Bajo overfitting)")

### 6.3. Validación cruzada con K-Fold (5 pliegues)

Para comprobar que el rendimiento del baseline no depende de cómo se hayan dividido los datos, se aplica validación cruzada con 5 pliegues. La idea es dividir el conjunto de entrenamiento en 5 partes, entrenar el modelo 5 veces usando cada vez una parte distinta como validación interna, y promediar los resultados.

Si los scores de los 5 folds son similares entre sí, el modelo es consistente. Si hay mucha variabilidad, significaría que el rendimiento depende demasiado de la partición concreta.

Se usa `KFold` con shuffle activado y semilla fija para que los resultados sean reproducibles.

In [ ]:
# Configurar KFold
kfold = KFold(n_splits=5, shuffle=True, random_state=SEED)

# Realizar cross-validation
print("Ejecutando validación cruzada con 5 folds...")
cv_scores = cross_val_score(
    model_forest_baseline,
    X_train,
    y_train,
    cv=kfold,
    scoring='accuracy',
    n_jobs=-1
)

print("Scores de cada fold:")
for i, score in enumerate(cv_scores, 1):
    bar = "█" * int(score * 50) + "░" * (50 - int(score * 50))
    print(f"  Fold {i}: {score:.4f} [{bar}]")

print(f"Media CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# Visualizar CV scores
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(1, len(cv_scores) + 1), cv_scores, color='steelblue', alpha=0.7)
ax.axhline(y=cv_scores.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {cv_scores.mean():.4f}')
ax.fill_between(np.arange(0.5, len(cv_scores) + 0.5 + 0.01, 0.01),
                cv_scores.mean() - cv_scores.std(),
                cv_scores.mean() + cv_scores.std(),
                alpha=0.2, color='red')
ax.set_xlabel('Fold', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Validación Cruzada - 5 Folds (Baseline)', fontweight='bold', fontsize=14)
ax.set_ylim([0, 1])
ax.legend()
plt.tight_layout()
plt.show()

### 6.4. Optimización de hiperparámetros con GridSearchCV

Visto el rendimiento del baseline, el siguiente paso es buscar una mejor configuración de hiperparámetros. Para ello se usa `GridSearchCV`, que prueba todas las combinaciones posibles de los valores definidos y selecciona la mejor según validación cruzada interna de 5 pliegues.

Los hiperparámetros explorados son:

| Hiperparámetro | Valores probados | Descripción |
|---|---|---|
| `n_estimators` | [100] | Número de árboles en el bosque |
| `max_depth` | [15, 25] | Profundidad máxima de cada árbol para controlar el sobreajuste |
| `min_samples_split` | [5] | Mínimo de muestras para dividir un nodo interno |
| `min_samples_leaf` | [2] | Mínimo de muestras en una hoja terminal |
| `max_features` | ['sqrt'] | Features a considerar en cada división |
| `class_weight` | ['balanced'] | Pondera las clases para compensar el desbalance |

La cuadrícula se ha mantenido reducida para no disparar el tiempo de cómputo. Se incluye `class_weight='balanced'` porque las clases de `CRASHSEVERITY` están muy desbalanceadas (la mayoría son accidentes sin heridos), y sin este ajuste el modelo tendería a ignorar las clases minoritarias.

Como métrica de optimización se usa **F1-Weighted** en lugar de accuracy porque, con clases tan desbalanceadas, la accuracy puede ser engañosa: un modelo que prediga siempre la clase mayoritaria puede tener un accuracy alto pero ser inútil para detectar los accidentes más graves.

In [ ]:
import time

# Grid reducido para no disparar el tiempo de computo
param_grid_fast = {
    'n_estimators': [100],              # Solo 1 valor
    'max_depth': [15, 25],              # Solo 2 valores
    'min_samples_split': [5],           # Solo 1 valor
    'min_samples_leaf': [2],            # Solo 1 valor
    'max_features': ['sqrt'],           # Solo 1 valor
    'class_weight': ['balanced']        # Solo 1 valor
}

# Calcular total de combinaciones
total_combinations = 1
for values in param_grid_fast.values():
    total_combinations *= len(values)

print(f"Parámetros a probar (GRID REDUCIDO):")
for param, values in param_grid_fast.items():
    print(f"  - {param}: {values}")

print(f"Total de combinaciones: {total_combinations}")
print(f"Total de entrenamientos (5-fold CV): {total_combinations * 5}")

start_time = time.time()

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=SEED, n_jobs=-1),
    param_grid=param_grid_fast,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1  # Reducido a 1 para menos output
)

grid_search.fit(X_train, y_train)

elapsed_time = time.time() - start_time

print(f"GridSearchCV completado en {elapsed_time:.2f} segundos ({elapsed_time/60:.2f} minutos)")

print("Mejores parametros:")
for param, value in grid_search.best_params_.items():
    print(f"  - {param}: {value}")

print(f"Mejor score CV F1-Weighted: {grid_search.best_score_:.4f}")

### 6.5. Evaluación del modelo optimizado

Una vez que GridSearchCV ha encontrado la mejor combinación de hiperparámetros, se recupera el modelo ya entrenado con esa configuración (`best_estimator_`) y se evalúa sobre los tres conjuntos.

Comparar train, test y validación permite detectar sobreajuste: si el train está muy por encima del test, el modelo ha memorizado los datos en lugar de aprender patrones generales.

Se calculan accuracy, precision, recall y F1-Score en su variante `weighted` para tener en cuenta el desbalance de clases. También se imprime el `classification_report` completo para ver cómo se comporta el modelo en cada nivel de severidad por separado, ya que puede haber diferencias importantes entre clases frecuentes (daños materiales) y clases raras (accidentes fatales).

In [ ]:
# Usar los mejores parámetros
model_forest_best = grid_search.best_estimator_

# Predicciones
y_pred_best_train = model_forest_best.predict(X_train)
y_pred_best_test = model_forest_best.predict(X_test)
y_pred_best_val = model_forest_best.predict(X_val)

# Métricas
acc_best_train = accuracy_score(y_train, y_pred_best_train)
acc_best_test = accuracy_score(y_test, y_pred_best_test)
prec_best_test = precision_score(y_test, y_pred_best_test, average='weighted', zero_division=0)
rec_best_test = recall_score(y_test, y_pred_best_test, average='weighted', zero_division=0)
f1_best_test = f1_score(y_test, y_pred_best_test, average='weighted', zero_division=0)
acc_best_val = accuracy_score(y_val, y_pred_best_val)

print("--- Metricas (test) ---")
print(f"Accuracy:  {acc_best_test:.4f}")
print(f"Precision: {prec_best_test:.4f}")
print(f"Recall:    {rec_best_test:.4f}")
print(f"F1-Score:  {f1_best_test:.4f}")

print("--- Classification report ---")
print(classification_report(y_test, y_pred_best_test, zero_division=0))

print("--- Accuracy train / test / val ---")
print(f"Train: {acc_best_train:.4f}")
print(f"Test:  {acc_best_test:.4f}")
print(f"Val:   {acc_best_val:.4f}")

### 6.6. Visualización de resultados

Se generan dos gráficas para interpretar mejor los resultados:

#### Matriz de confusión
Muestra cuántas instancias de cada clase real han sido clasificadas como cada clase predicha. Es útil para ver en qué categorías de severidad falla más el modelo y si hay confusiones sistemáticas entre clases concretas (p.ej. entre accidentes con lesiones posibles y accidentes con daños materiales únicamente).

#### Importancia de características (*Feature Importance*)
Muestra las 15 variables que más influyen en las predicciones. En Random Forest esta importancia se calcula a partir de cuánto reduce la impureza Gini cada variable en promedio a lo largo de todos los árboles. Sirve para comprobar si los factores más relevantes tienen sentido desde el punto de vista de la seguridad vial, e identificar si hay variables que aportan poco y podrían descartarse.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Matriz de confusión
cm = confusion_matrix(y_test, y_pred_best_test)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False)
axes[0].set_title('Matriz de Confusión (TEST SET)', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Predicción')
axes[0].set_ylabel('Real')

# 2. Feature Importance (Top 15)
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model_forest_best.feature_importances_
}).sort_values('importance', ascending=False)

axes[1].barh(feature_importance['feature'].head(15),
             feature_importance['importance'].head(15),
             color='steelblue')
axes[1].set_xlabel('Importancia')
axes[1].set_title('Top 15 Features Más Importantes', fontweight='bold', fontsize=12)
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

### 6.7. Comparativa Baseline vs. Optimizado (Random Forest)

Se comparan el modelo sin ajustar y el modelo optimizado sobre el conjunto de test, para ver si la búsqueda de hiperparámetros ha valido la pena. Tanto la tabla como la gráfica de barras muestran las diferencias en las cuatro métricas principales entre ambas versiones.

> **Nota**: La comparativa entre los tres modelos del trabajo (Random Forest, Modelo 2 y Modelo 3) se hará en la **sección 9** cuando estén todos implementados.

In [ ]:
comparativa = pd.DataFrame({
    'Métrica': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Baseline': [
        accuracy_score(y_test, y_pred_baseline_test),
        precision_score(y_test, y_pred_baseline_test, average='weighted', zero_division=0),
        recall_score(y_test, y_pred_baseline_test, average='weighted', zero_division=0),
        f1_score(y_test, y_pred_baseline_test, average='weighted', zero_division=0)
    ],
    'Optimizado': [acc_best_test, prec_best_test, rec_best_test, f1_best_test]
})

print(comparativa.to_string(index=False))

# Gráfica comparativa
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(comparativa))
width = 0.35

bars1 = ax.bar(x - width/2, comparativa['Baseline'], width, label='Baseline', color='lightcoral', alpha=0.8)
bars2 = ax.bar(x + width/2, comparativa['Optimizado'], width, label='Optimizado', color='steelblue', alpha=0.8)

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Comparativa: Baseline vs Optimizado', fontweight='bold', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(comparativa['Métrica'])
ax.legend()
ax.set_ylim([0, 1])

# Valores en barras
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# Calcular mejora
mejora = ((acc_best_test - accuracy_score(y_test, y_pred_baseline_test)) /
          accuracy_score(y_test, y_pred_baseline_test) * 100)
print(f"Mejora en Accuracy: {mejora:+.2f}%")

### 6.8. Resumen del modelo Random Forest

El bloque siguiente imprime un resumen con los resultados finales del Random Forest optimizado: métricas en train, test y validación, estabilidad de la validación cruzada, la mejora respecto al baseline y las variables más importantes. Estos valores se usarán en la sección 9 para comparar los tres modelos entre sí y seleccionar el mejor.

In [ ]:
resumen_final = pd.DataFrame({
    'Conjunto': ['Train (CV Mean)', 'Test', 'Validación'],
    'Accuracy': [cv_scores.mean(), acc_best_test, acc_best_val],
    'Precision': [np.nan, prec_best_test,
                 precision_score(y_val, y_pred_best_val, average='weighted', zero_division=0)],
    'Recall': [np.nan, rec_best_test,
              recall_score(y_val, y_pred_best_val, average='weighted', zero_division=0)],
    'F1-Score': [np.nan, f1_best_test,
                f1_score(y_val, y_pred_best_val, average='weighted', zero_division=0)]
})

print(resumen_final.to_string())

print(f"""
{'='*70}
Rendimiento:
  - Accuracy (Test):  {acc_best_test:.4f}
  - F1-Score (Test):  {f1_best_test:.4f}
  - Desv. std CV:     {cv_scores.std():.4f}

Overfitting:
  - Gap Train-Test: {abs(acc_best_train - acc_best_test):.4f}  {'-> bajo overfitting' if abs(acc_best_train - acc_best_test) < 0.05 else '-> overfitting moderado, revisar regularizacion'}

Top 3 variables mas importantes:
{feature_importance.head(3).to_string(index=False)}

Accuracy por conjunto:
  - Train:      {acc_best_train:.4f}
  - Test:       {acc_best_test:.4f}
  - Validacion: {acc_best_val:.4f}
  {'-> resultados consistentes entre conjuntos' if abs(acc_best_test - acc_best_val) < 0.02 else '-> diferencias entre test y validacion, revisar particion'}

Mejora sobre baseline:
  - Baseline -> Optimizado: {mejora:+.2f}%

Notas:
  {'-> accuracy > 0.80, resultados aceptables' if acc_best_test > 0.80 else '-> accuracy < 0.80, considerar mas datos o ajustes'}
  {'-> importancia de features correcta' if feature_importance.iloc[0]['importance'] > 0.05 else '-> revisar importancia de features'}
  {'-> el modelo predice todas las clases' if len(np.unique(y_pred_best_test)) == len(np.unique(y_test)) else '-> el modelo ignora alguna clase, revisar desbalance'}
{'='*70}
""")


## 7. Modelo 2: [Pendiente de implementar]

Los resultados obtenidos se integrarán en la comparativa de la **sección 9**.

## 8. Modelo 3: [Pendiente de implementar]

Los resultados obtenidos se integrarán en la comparativa de la **sección 9**.

## 9. Comparativa de los 3 modelos

> **Esta sección se completará una vez estén implementados los tres modelos (secciones 6, 7 y 8).**

En esta sección se realizará una **comparativa directa** entre los tres modelos de Machine Learning desarrollados, evaluando el rendimiento de cada uno sobre el mismo conjunto de test para garantizar la equidad de la comparación.

La comparativa incluirá:

- **Tabla comparativa de métricas**: Accuracy, Precision, Recall y F1-Score (weighted) de los tres modelos sobre el conjunto de test.
- **Gráfica de barras comparativa**: Visualización del rendimiento relativo de cada modelo.
- **Análisis de overfitting**: Comparación del gap train-test en los tres modelos.
- **Análisis de estabilidad**: Comparación de la variabilidad en validación cruzada (desviación estándar de los *folds*).
- **Análisis de coste computacional**: Comparación del tiempo de entrenamiento e inferencia.

Los resultados de esta comparativa orientarán la **selección del modelo final** en la sección 10.

## 10. Selección del modelo final y conclusiones generales

> **Esta sección se completará una vez estén implementados los tres modelos (secciones 6, 7 y 8) y realizada la comparativa (sección 9).**

Una vez evaluados los tres modelos de Machine Learning de forma individual y comparados entre sí en la sección 9, esta sección final recoge:

- **Selección razonada del mejor modelo**: Justificación de qué modelo se selecciona como modelo final, atendiendo a las métricas de rendimiento, la capacidad de generalización, la interpretabilidad y el coste computacional.
- **Evaluación final del modelo seleccionado**: Análisis en profundidad del modelo elegido sobre el conjunto de validación, que hasta este punto ha permanecido completamente aislado del proceso de entrenamiento y optimización.
- **Conclusiones del trabajo**: Reflexión global sobre el proceso completo seguido, los aprendizajes obtenidos, las limitaciones encontradas y las posibles líneas de mejora futura.

### Factores de selección del modelo

A la hora de seleccionar el modelo final se tendrán en cuenta los siguientes criterios:

| Criterio | Descripción |
|---|---|
| **Rendimiento** | F1-Weighted sobre el conjunto de test |
| **Generalización** | Gap entre train y test (overfitting) |
| **Estabilidad** | Desviación estándar en validación cruzada |
| **Interpretabilidad** | Capacidad de explicar las predicciones |
| **Coste computacional** | Tiempo de entrenamiento e inferencia |